# ViralFutbol Studio — Kalıcı ve Kurtarılabilir Colab

Bu defter MoneyPrinterTurbo'yu doğrulanmış sürümden kurar ve ViralFutbol dayanıklılık paketini uygular.

- Ayarlar, API anahtarları, görev durumu ve videolar Google Drive'da kalır.
- Yarım/bozuk MP4 başarılı sayılmaz; tamamlanan video yeniden başlatma sonrası geri bulunur.
- Aynı haber yeniden gönderilirse mevcut Drive görevi döner; ikinci video başlamaz.
- Arayüz, API ve video indirme tek sabit ngrok adresinden çalışır.
- Hızlı üretim profili 720×1280, H.264/AAC ve dinamik geçişler kullanır.
- YouTube'a otomatik yükleme kapalıdır.

Hücreleri yukarıdan aşağıya birer kez çalıştırın.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import gzip
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/MoneyPrinterTurbo-ViralFutbol")
BASE_REF = "v1.3.6"
PATCH_GZ_B64 = "..."

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BASE_REF, "--depth", "1",
         "https://github.com/harry0703/MoneyPrinterTurbo.git", str(REPO)],
        check=True,
    )
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"{REPO} var ama geçerli bir Git deposu değil")

subprocess.run(["git", "-C", str(REPO), "reset", "--hard", BASE_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "clean", "-fd"], check=True)
for generated_test in (
    "test_persistent_tasks.py",
    "test_viral_futbol.py",
    "test_material_download.py",
    "test_idempotent_tasks.py",
):
    (REPO / "tests" / generated_test).unlink(missing_ok=True)

patch_file = Path("/content/viral-futbol-persistence.patch")
patch_file.write_bytes(gzip.decompress(base64.b64decode(PATCH_GZ_B64)))
subprocess.run(["git", "-C", str(REPO), "apply", "--check", str(patch_file)], check=True)
subprocess.run(["git", "-C", str(REPO), "apply", str(patch_file)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    ["uv", "sync", "--project", str(REPO), "--no-dev", "--python", "3.11"],
    check=True,
)
VENV_PYTHON = REPO / ".venv" / "bin" / "python"
subprocess.run(
    ["uv", "pip", "install", "--python", str(VENV_PYTHON), "pyngrok"],
    check=True,
)
print("✅ Kalıcı ViralFutbol üretim motoru kuruldu.")


## 2. Anahtarları Drive'a bir kez kaydet

Kutular gizlidir. Daha önce kaydettiğiniz bir değeri korumak için ilgili kutuyu boş bırakabilirsiniz. Anahtarları sohbet mesajına veya GitHub'a yazmayın.

In [ ]:
import json
import os
import shutil
import subprocess
from getpass import getpass
from pathlib import Path

PERSISTENT_ROOT = Path("/content/drive/MyDrive/ViralFutbol")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_FILE = PERSISTENT_ROOT / "config.toml"
RUNTIME_SECRETS = PERSISTENT_ROOT / "runtime-secrets.json"

if not CONFIG_FILE.exists():
    shutil.copyfile(REPO / "config.example.toml", CONFIG_FILE)

saved = {}
if RUNTIME_SECRETS.exists():
    try:
        saved = json.loads(RUNTIME_SECRETS.read_text(encoding="utf-8"))
    except Exception:
        saved = {}

def keep_or_prompt(key, label, required=True):
    value = getpass(f"{label} (boş = kayıtlı değeri koru): ").strip()
    if value:
        saved[key] = value
    if required and not saved.get(key):
        raise ValueError(f"{label} gerekli")

keep_or_prompt("openrouter_api_key", "OpenRouter API anahtarı")
keep_or_prompt("pexels_api_key", "Pexels API anahtarı")
keep_or_prompt("mpt_api_key", "MoneyPrinterTurbo ortak API anahtarı")
keep_or_prompt("ngrok_authtoken", "ngrok authtoken")

old_domain = saved.get("ngrok_domain", "")
domain = input(f"Sabit ngrok alan adı [{old_domain or 'ör. adiniz.ngrok-free.dev'}]: ").strip()
if domain:
    domain = domain.removeprefix("https://").removeprefix("http://").strip("/")
    saved["ngrok_domain"] = domain
if not saved.get("ngrok_domain"):
    raise ValueError("Sabit ngrok alan adı gerekli")

update_script = r'''\
import json, os, sys, tempfile
from pathlib import Path
import toml
payload = json.load(sys.stdin)
target = Path(payload["config_file"])
data = toml.load(target)
app = data.setdefault("app", {})
app["llm_provider"] = "openrouter"
app["openrouter_api_key"] = payload["openrouter_api_key"]
app["openrouter_base_url"] = "https://openrouter.ai/api/v1"
app["openrouter_model_name"] = "minimax/minimax-m3:free"
app["pexels_api_keys"] = [payload["pexels_api_key"]]
app["api_key"] = payload["mpt_api_key"]
app["state_backend"] = "sqlite"
app["state_db_path"] = ""
app["video_output_quality"] = "720p"
app["video_preset"] = "veryfast"
fd, temp_name = tempfile.mkstemp(prefix=".config-", suffix=".toml", dir=target.parent)
try:
    with os.fdopen(fd, "w", encoding="utf-8") as stream:
        toml.dump(data, stream)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temp_name, target)
except Exception:
    try: os.unlink(temp_name)
    except OSError: pass
    raise
'''
payload = {
    "config_file": str(CONFIG_FILE),
    "openrouter_api_key": saved["openrouter_api_key"],
    "pexels_api_key": saved["pexels_api_key"],
    "mpt_api_key": saved["mpt_api_key"],
}
subprocess.run(
    [str(VENV_PYTHON), "-c", update_script],
    input=json.dumps(payload), text=True, check=True,
)
RUNTIME_SECRETS.write_text(json.dumps(saved, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
os.chmod(RUNTIME_SECRETS, 0o600)
print("✅ Anahtarlar ve sabit alan adı Google Drive'a kalıcı kaydedildi.")


## 3. Ücretsiz gerçek üretim testi

Bu hücre dış API harcaması yapmadan renkli kaynak klip, ses ve Türkçe altyazı üretir; final MP4'ün görüntü, ses, çözünürlük ve yeniden başlatma kurtarmasını doğrular.

In [ ]:
import subprocess
from datetime import datetime

test_id = "colab-e2e-" + datetime.utcnow().strftime("%Y%m%d-%H%M%S")
subprocess.run(
    [str(VENV_PYTHON), str(REPO / "scripts" / "verify_persistent_pipeline.py"),
     "--storage", str(PERSISTENT_ROOT / "storage"),
     "--task-id", test_id, "--quality", "720p"],
    cwd=REPO,
    check=True,
)
print("✅ Gerçek MP4 testi geçti. Dosya:")
print(PERSISTENT_ROOT / "storage" / "tasks" / test_id / "final-1.mp4")


## 4. Arayüz + API + sabit ngrok bağlantısını başlat

Hücre URL'leri yazınca tamamlanır; arka plandaki süreç Colab oturumu açık kaldığı sürece çalışır. Aynı sabit alan adı kullanıldığı için Render adresini her yeniden başlatmada değiştirmek gerekmez.

In [ ]:
import json
import os
import select
import subprocess
import time

saved = json.loads(RUNTIME_SECRETS.read_text(encoding="utf-8"))
environment = os.environ.copy()
environment["NGROK_AUTHTOKEN"] = saved["ngrok_authtoken"]
environment["NGROK_DOMAIN"] = saved["ngrok_domain"]
environment["MPT_VIDEO_OUTPUT_QUALITY"] = "720p"

MPT_LAUNCHER_PROCESS = subprocess.Popen(
    [str(VENV_PYTHON), str(REPO / "scripts" / "colab_persistent_launcher.py"),
     "--drive-root", str(PERSISTENT_ROOT),
     "--domain", saved["ngrok_domain"]],
    cwd=REPO,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
MPT_PUBLIC_URL = ""
MPT_API_URL = ""
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if MPT_LAUNCHER_PROCESS.poll() is not None:
        remainder = MPT_LAUNCHER_PROCESS.stdout.read()
        raise RuntimeError("Başlatıcı durdu:\n" + remainder)
    readable, _, _ = select.select([MPT_LAUNCHER_PROCESS.stdout], [], [], 1)
    if not readable:
        continue
    line = MPT_LAUNCHER_PROCESS.stdout.readline()
    if not line:
        continue
    print(line.rstrip())
    if line.startswith("WebUI:"):
        MPT_PUBLIC_URL = line.split(":", 1)[1].strip()
    elif line.startswith("MoneyPrinterTurbo API:"):
        MPT_API_URL = line.split(":", 1)[1].strip()
    if MPT_PUBLIC_URL and MPT_API_URL:
        break
else:
    raise TimeoutError("Servisler 180 saniyede hazır olmadı; Drive/ViralFutbol/logs dosyalarını kontrol edin")

print("✅ Tek adresli kalıcı bağlantı hazır.")


## 5. Canlı bağlantıyı doğrula

Başarılı sonuçtan sonra Render'da MPT_API_URL değerini yukarıdaki API adresi, MPT_API_KEY değerini de aynı ortak anahtar yapın. İlk bağlantıda NEWS_DRY_RUN=true kalmalıdır. Bu defter video yayımlamaz.

In [ ]:
import requests

headers = {"x-api-key": saved["mpt_api_key"]}
ui = requests.get(MPT_PUBLIC_URL + "/_stcore/health", timeout=20)
tasks = requests.get(MPT_API_URL + "/tasks?page=1&page_size=1", headers=headers, timeout=20)
ui.raise_for_status()
tasks.raise_for_status()
print("✅ UI:", ui.text.strip())
print("✅ API: görev listesi erişilebilir")
print("Render MPT_API_URL =", MPT_API_URL)
print("MPT_API_KEY ekranda gösterilmedi; Drive'daki kayıtlı ortak anahtarı kullanın.")
